# TPF Super-Resolución — trabajo completo (4 ejes)

**Antes de correr (panel derecho):**
- Accelerator = **GPU T4**
- **Internet = ON**  (necesario para bajar la VGG19 de la pérdida perceptual / SRGAN)
- **Add Input** con tu dataset `tpf-superres` (versión nueva, con el código actualizado)

Recomendado: **Save Version → Save & Run All (Commit)** para que se entrene en el servidor y persista todo.
Cada celda es un eje; podés comentar lo que no quieras. El entrenamiento completo es ~2-3 h en T4.

In [ ]:
# 1) Copiar el proyecto a /kaggle/working (escribible)
import os, shutil
proj = None
for root, dirs, _ in os.walk('/kaggle/input'):
    if 'src' in dirs and 'datasets' in dirs:
        proj = root; break
assert proj, 'No encontre src/ y datasets/ en /kaggle/input'
print('Proyecto en:', proj)
for sub in ['src', 'datasets']:
    dst = f'/kaggle/working/{sub}'
    if not os.path.exists(dst):
        shutil.copytree(os.path.join(proj, sub), dst)
os.makedirs('/kaggle/working/outputs', exist_ok=True)
print(os.listdir('/kaggle/working'))

In [ ]:
# 2) Sanity: GPU + gate de evaluacion
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
%cd /kaggle/working/src
!python metrics_check.py

In [ ]:
# 3) EJE 1 — Progresion de arquitecturas (SRCNN -> FSRCNN -> +residual -> EDSR) en x2 y x4
%cd /kaggle/working/src
EP='--epochs 150 --patches-per-epoch 8000 --val-every 10'
for scale in [2, 4]:
    for model in ['srcnn', 'fsrcnn', 'fsrcnn_res']:
        !python train.py --model {model} --scale {scale} --batch 64 {EP}
    !python train.py --model edsr --scale {scale} --batch 16 {EP}

In [ ]:
# 4) EJE 4 — Completitud: factor 3x + ablacion de perdidas (L1 ya esta; agregamos L2 y Charbonnier)
%cd /kaggle/working/src
EP='--epochs 150 --patches-per-epoch 8000 --val-every 10'
for model in ['fsrcnn', 'fsrcnn_res']:
    !python train.py --model {model} --scale 3 --batch 64 {EP}
!python train.py --model edsr --scale 3 --batch 16 {EP}
for loss in ['l2', 'charbonnier']:
    !python train.py --model fsrcnn --scale 2 --loss {loss} --batch 64 {EP}

In [ ]:
# 5) EJE 2 — Distorsion vs percepcion: perceptual (VGG) y SRGAN, afinando desde EDSR x4 L1
#    (requiere Internet ON; usa outputs/edsr_x4_l1/best.pth de la celda 3)
%cd /kaggle/working/src
!python train.py --model edsr --scale 4 --loss perceptual --init ../outputs/edsr_x4_l1/best.pth --epochs 120 --patches-per-epoch 8000 --val-every 10 --batch 16
!python train_gan.py --gen edsr --scale 4 --init ../outputs/edsr_x4_l1/best.pth --epochs 120 --patches-per-epoch 8000 --val-every 10 --batch 16

In [ ]:
# 6) Tablas + figuras + benchmark de eficiencia (Eje 3)
%cd /kaggle/working/src
!python make_figures.py --scales 2 3 4 --testsets Set5 Set14 BSD100 Urban100 --panels 0 2
!python bench.py --scales 2 3 4

In [ ]:
# 7) Comprimir outputs y mostrar resumen
import shutil, json, glob
shutil.make_archive('/kaggle/working/outputs', 'zip', '/kaggle/working/outputs')
print('outputs.zip listo')
for h in sorted(glob.glob('/kaggle/working/outputs/*/history.json')):
    d = json.load(open(h))
    print(h.split('/')[-2], '->', d.get('final'))

Al terminar el Commit, los pesos y figuras quedan en `outputs/` (y `outputs.zip`).
Se descargan desde la pestaña **Output** de la versión, o por API.